# IDRiD retinal lesion segmentation
Train U-Net, U-Net++ or MAnet for microaneurysm, hemorrhage and hard-exudate masks. Use a Colab GPU runtime and run cells from top to bottom.

In [ ]:
import os, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
repo_dir = '/content/dr-diagnostic-system'
repo_url = 'https://github.com/Bang334/dr-diagnostic-system.git'
branch = 'feat/merged-dataset-training'
if not os.path.isdir(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', repo_url, repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/segmentation/requirements.txt'], check=True)
os.environ['TORCH_HOME'] = '/content/drive/MyDrive/torch_cache'
print('Code and dependencies are ready.')

## Select the experiment
The suitable starter dataset is already selected. The first download is cached in Drive; later runs reuse it.

In [ ]:
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display
architecture = widgets.ToggleButtons(
    options=[('U-Net (baseline)', 'unet'), ('U-Net++', 'unetplusplus'), ('MAnet / attention', 'manet')],
    value='unet', description='Model:'
)
dataset_slug = widgets.Text(
    value='dankok/diabetic-retinopathy-image-dataset',
    description='Kaggle:', layout=widgets.Layout(width='850px'), disabled=True
)
run_name = widgets.Text(
    value='idrid_lesions_' + datetime.now().strftime('%Y%m%d_%H%M'),
    description='Run:', layout=widgets.Layout(width='600px')
)
display(architecture, dataset_slug, run_name)
print('Change the model/run name if needed; no Enter key is required.')

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import userdata
from getpass import getpass
try:
    token = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    token = getpass('Kaggle API token (hidden): ').strip()
if not token:
    raise RuntimeError('Kaggle API token is required to download the selected dataset.')
os.environ['KAGGLE_API_TOKEN'] = token
cache_dir = Path('/content/drive/MyDrive/retfound_datasets/idrid_segmentation')
extract_dir = Path('/content/selected_data/idrid_segmentation')
cache_dir.mkdir(parents=True, exist_ok=True)
zip_path = cache_dir / 'diabetic-retinopathy-image-dataset.zip'
if not zip_path.is_file():
    subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_slug.value, '-p', str(cache_dir)], check=True)
    downloaded = next(cache_dir.glob('*.zip'))
    if downloaded != zip_path:
        downloaded.replace(zip_path)
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as archive:
    segmentation_members = [name for name in archive.namelist() if 'Segmentation/' in name.replace('\\', '/')]
    if not segmentation_members:
        raise RuntimeError('The downloaded archive does not contain the IDRiD Segmentation directory.')
    for member in segmentation_members:
        archive.extract(member, extract_dir)
print('IDRiD segmentation files extracted to:', extract_dir)

In [ ]:
from ai.segmentation.data import LESION_NAMES, build_idrid_manifest
manifest = build_idrid_manifest(extract_dir)
print('Images:', len(manifest), manifest['official_split'].value_counts().to_dict())
for lesion in LESION_NAMES:
    print(lesion, 'mask files:', int((manifest[f'{lesion}_mask'] != '').sum()))
display(manifest.head())

In [ ]:
output_dir = Path('/content/drive/MyDrive/retfound_segmentation') / run_name.value / architecture.value
cmd = [
    sys.executable, '-m', 'ai.segmentation.train',
    '--dataset-dir', str(extract_dir), '--output-dir', str(output_dir),
    '--architecture', architecture.value, '--encoder-name', 'resnet34',
    '--encoder-weights', 'imagenet', '--image-size', '768',
    '--epochs', '40', '--freeze-epochs', '2', '--patience', '8',
    '--batch-size', '2', '--accum-steps', '2',
    '--encoder-lr', '1e-4', '--decoder-lr', '3e-4',
    '--num-workers', '2', '--seed', '42'
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from IPython.display import Image, display
summary = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
for preview in sorted((output_dir / 'previews').glob('*.png')):
    display(Image(filename=str(preview), width=900))

## Fair comparison
Create a new run name and rerun the training cell for U-Net++ or MAnet. Keep seed, split, encoder, resolution and loss unchanged. Compare macro Dice/IoU and per-lesion recall in `summary.json`; do not choose a model from test results repeatedly.